# GraphRAG Query Pipeline — See How Map-Reduce Works

This notebook lets you ask a question and see **every step** of the GraphRAG global search pipeline:
1. Which community reports were selected
2. Each MAP response (key points + importance scores)
3. The REDUCE step (final synthesized answer)

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Load API key
load_dotenv()
client = OpenAI(api_key=os.getenv('GRAPHRAG_API_KEY'))

MODEL = 'gpt-4.1'

print(f'Using model: {MODEL}')
print('API key loaded:', 'Yes' if os.getenv('GRAPHRAG_API_KEY') else 'No')

Using model: gpt-4.1
API key loaded: Yes


In [ ]:
# Load the GraphRAG output files
df_entities = pd.read_parquet('output/entities.parquet')
df_relationships = pd.read_parquet('output/relationships.parquet')
df_communities = pd.read_parquet('output/communities.parquet')
df_reports = pd.read_parquet('output/community_reports.parquet')

print(f'Entities: {len(df_entities)}')
print(f'Relationships: {len(df_relationships)}')
print(f'Communities: {len(df_communities)}')
print(f'Community Reports: {len(df_reports)}')
print(f'\nCommunity Levels: {sorted(df_communities["level"].unique())}')
print(f'Reports per level:')
print(df_reports.groupby('level').size())

Entities: 209
Relationships: 316
Communities: 39
Community Reports: 39

Community Levels: [np.int64(0), np.int64(1)]
Reports per level:
level
0    11
1    28
dtype: int64


In [ ]:
# Choose community level (higher = more granular sub-communities)
COMMUNITY_LEVEL = 1  # Change this to 0 for top-level, or higher if available

# Filter reports at or below the chosen level
reports_at_level = df_reports[df_reports['level'] <= COMMUNITY_LEVEL].copy()
print(f'Using {len(reports_at_level)} community reports at level <= {COMMUNITY_LEVEL}')
print(f'\nReport titles:')
for _, r in reports_at_level.iterrows():
    print(f'  [Level {r["level"]}] Community {r["community"]}: {r["title"][:80]}...')

Using 39 community reports at level <= 1

Report titles:
  [Level 1] Community 11: Multicast Authentication and Secure Network Protocols Research Community (Tygar,...
  [Level 1] Community 12: A. Perrig and Collaborators in Secure Wireless Protocols (Ariadne, SPINS, and MO...
  [Level 1] Community 13: Hash Sequence Cryptography Research and Financial Cryptography Events...
  [Level 1] Community 14: IEEE Symposium on Research in Security and Privacy Community...
  [Level 1] Community 15: TESLA Protocol and Secure Broadcast Authentication Community...
  [Level 1] Community 16: TESLA Protocol Research Community: UC Berkeley and Key Contributors...
  [Level 1] Community 17: Broadcast Distribution Networks and Authentication Protocols...
  [Level 1] Community 18: Clock Synchronization and Drift in the TESLA Protocol...
  [Level 1] Community 19: Ran Canetti and IBM Research: TESLA Protocol Collaboration...
  [Level 1] Community 20: TESLA Protocol: Network Packet and Message Structure...
  [L

In [ ]:
# ===== THE ACTUAL PROMPTS (same as Microsoft GraphRAG uses) =====

MAP_SYSTEM_PROMPT = """
---Role---
You are a helpful assistant responding to questions about data in the tables provided.

---Goal---
Generate a response consisting of a list of key points that responds to the user's question,
summarizing all relevant information in the input data tables.

If you don't know the answer or if the input data tables do not contain sufficient information
to provide an answer, just say so. Do not make anything up.

Each key point in the response should have the following element:
- Description: A comprehensive description of the point.
- Importance Score: An integer score between 0-100 that indicates how important the point is
  in answering the user's question. An 'I don't know' type of response should have a score of 0.

The response should be JSON formatted as follows:
{{
    "points": [
        {{"description": "Description of point 1 [Data: Reports (report ids)]", "score": score_value}},
        {{"description": "Description of point 2 [Data: Reports (report ids)]", "score": score_value}}
    ]
}}

Do not include information where the supporting evidence for it is not provided.

---Data tables---

{context_data}
"""

REDUCE_SYSTEM_PROMPT = """
---Role---
You are a helpful assistant responding to questions about a dataset by synthesizing
perspectives from multiple analysts.

---Goal---
Generate a response of the target length and format that responds to the user's question,
summarize all the reports from multiple analysts who focused on different parts of the dataset.

Note that the analysts' reports provided below are ranked in the descending order of importance.

If you don't know the answer or if the provided reports do not contain sufficient information
to provide an answer, just say so. Do not make anything up.

The final response should remove all irrelevant information from the analysts' reports and merge
the cleaned information into a comprehensive answer that provides explanations of all the key
points and implications appropriate for the response length and format.

Add sections and commentary to the response as appropriate for the length and format.
Style the response in markdown.

---Target response length and format---
Multiple paragraphs

---Analyst Reports---

{report_data}
"""

print('Prompts loaded.')

Prompts loaded.


In [ ]:
def run_map_phase(question, reports_df, batch_size=5):
    """
    MAP PHASE: Send each batch of community reports to the LLM.
    Returns a list of (batch_index, key_points) tuples.
    """
    all_map_results = []
    
    # Batch the reports
    report_batches = []
    for i in range(0, len(reports_df), batch_size):
        batch = reports_df.iloc[i:i+batch_size]
        # Build context: report summaries as a table
        context_rows = []
        for _, row in batch.iterrows():
            context_rows.append(
                f"Report {row['community']}: {row['title']}\n"
                f"Summary: {row['summary']}\n"
                f"Full Content: {row['full_content'][:2000]}\n"
            )
        report_batches.append('\n---\n'.join(context_rows))
    
    print(f'MAP PHASE: Sending {len(report_batches)} batches to LLM...\n')
    
    for i, batch_context in enumerate(report_batches):
        print(f'  Processing batch {i+1}/{len(report_batches)}...')
        
        prompt = MAP_SYSTEM_PROMPT.format(context_data=batch_context)
        
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': prompt},
                {'role': 'user', 'content': question}
            ],
            response_format={'type': 'json_object'},
            temperature=0
        )
        
        raw = response.choices[0].message.content
        try:
            parsed = json.loads(raw)
            points = parsed.get('points', [])
        except json.JSONDecodeError:
            points = []
        
        all_map_results.append({
            'batch': i + 1,
            'points': points,
            'raw_response': raw
        })
    
    return all_map_results

print('Map function defined.')

Map function defined.


In [17]:
import tiktoken

def run_reduce_phase(question, map_results, max_reduce_tokens=8000):
    """
    REDUCE PHASE: Collect all key points, sort by score,
    truncate to token limit (like original GraphRAG), and synthesize.
    """
    enc = tiktoken.encoding_for_model(MODEL)
    
    # Collect and sort all key points
    all_points = []
    for result in map_results:
        for point in result['points']:
            all_points.append({
                'batch': result['batch'],
                'description': point.get('description', ''),
                'score': point.get('score', 0)
            })
    
    # Filter out score=0 and sort descending
    filtered_points = [p for p in all_points if p['score'] > 0]
    filtered_points.sort(key=lambda x: x['score'], reverse=True)
    
    print(f'REDUCE PHASE: {len(all_points)} total points, {len(filtered_points)} after filtering (score > 0)')
    
    if not filtered_points:
        return 'I am sorry but I am unable to answer this question given the provided data.', filtered_points
    
    # Format as analyst reports WITH TOKEN LIMIT (like original GraphRAG)
    report_sections = []
    total_tokens = 0
    included = 0
    for p in filtered_points:
        section = (
            f"----Analyst {p['batch']}----\n"
            f"Importance Score: {p['score']}\n"
            f"{p['description']}"
        )
        section_tokens = len(enc.encode(section))
        if total_tokens + section_tokens > max_reduce_tokens:
            print(f'  Token limit reached! Included {included}/{len(filtered_points)} points ({total_tokens} tokens)')
            break
        report_sections.append(section)
        total_tokens += section_tokens
        included += 1
    else:
        print(f'  All {included} points fit within token limit ({total_tokens}/{max_reduce_tokens} tokens)')
    
    report_data = '\n\n'.join(report_sections)
    
    # Send to LLM for final synthesis
    prompt = REDUCE_SYSTEM_PROMPT.format(report_data=report_data)
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': prompt},
            {'role': 'user', 'content': question}
        ],
        temperature=0
    )
    
    final_answer = response.choices[0].message.content
    return final_answer, filtered_points

print('Reduce function defined.')

Reduce function defined.


In [ ]:
# ============================================================
# ASK YOUR QUESTION HERE
# ============================================================

QUESTION = "What is TESLA and how does it provide broadcast authentication?"

print(f'Question: {QUESTION}')
print('=' * 60)

Question: What is TESLA and how does it provide broadcast authentication?


In [12]:
reports_at_level

,id,human_readable_id,community,level,parent,children,title,summary,full_content,rank,rating_explanation,findings,full_content_json,period,size
0,c78ea93fd19578d28622863de0e1e25e99a99d1287af0c...,11,11,1,0,[],Multicast Authentication and Secure Network Pr...,This community is composed of prominent academ...,# Multicast Authentication and Secure Network ...,8.5,The community’s pioneering work on multicast a...,[{'explanation': 'J. D. Tygar has been a key f...,"{\n ""title"": ""Multicast Authentication and ...",2026-02-19,6
1,e3202190f8e1698ce925be8ab982be93669f13cc151f26...,12,12,1,0,[],A. Perrig and Collaborators in Secure Wireless...,"This community centers around A. Perrig, a pro...",# A. Perrig and Collaborators in Secure Wirele...,8.5,The community's contributions to secure networ...,[{'explanation': 'A. Perrig emerges as the cen...,"{\n ""title"": ""A. Perrig and Collaborators i...",2026-02-19,7
2,79d5e6cca1f1f3521f591bf9d5b702cc12d2463fda3416...,13,13,1,0,[],Hash Sequence Cryptography Research and Financ...,This community consists of leading researchers...,# Hash Sequence Cryptography Research and Fina...,6.5,The impact severity rating reflects the commun...,[{'explanation': 'M. Jakobsson stands out in t...,"{\n ""title"": ""Hash Sequence Cryptography Re...",2026-02-19,3
3,1b5d0894f768594c06cd12a2cebad8c869f3d6d23e8963...,14,14,1,0,[],IEEE Symposium on Research in Security and Pri...,This community centers on the IEEE Symposium o...,# IEEE Symposium on Research in Security and P...,9.0,"The community, represented by the IEEE Symposi...",[{'explanation': 'The IEEE Symposium on Resear...,"{\n ""title"": ""IEEE Symposium on Research in...",2026-02-19,5
4,f0304ff64b8a81883a7792d4aab2955d31fddcd712f997...,15,15,1,1,[],TESLA Protocol and Secure Broadcast Authentica...,This community centers around the TESLA protoc...,# TESLA Protocol and Secure Broadcast Authenti...,8.5,TESLA and its associated community represent a...,[{'explanation': 'TESLA (Timed Efficient Strea...,"{\n ""title"": ""TESLA Protocol and Secure Bro...",2026-02-19,25
5,a53b211edb6819294da12f86611f20b64b29baae173946...,16,16,1,1,[],TESLA Protocol Research Community: UC Berkeley...,This report examines the community centered on...,# TESLA Protocol Research Community: UC Berkel...,7.0,The impact severity rating is high due to the ...,[{'explanation': 'The University of California...,"{\n ""title"": ""TESLA Protocol Research Commu...",2026-02-19,3
6,8fb7f3a2fd3b28df22f62345b0827b62a872c76b28ec4c...,17,17,1,1,[],Broadcast Distribution Networks and Authentica...,This community comprises entities central to t...,# Broadcast Distribution Networks and Authenti...,8.0,The impact severity rating is high because bro...,[{'explanation': 'Broadcast distribution netwo...,"{\n ""title"": ""Broadcast Distribution Networ...",2026-02-19,4
7,05c7774e6a69c422cb8137602493e413f9367235328217...,18,18,1,1,[],Clock Synchronization and Drift in the TESLA P...,This community is centered around the critical...,# Clock Synchronization and Drift in the TESLA...,8.0,The community is of high impact because precis...,"[{'explanation': 'Within the TESLA protocol, t...","{\n ""title"": ""Clock Synchronization and Dri...",2026-02-19,2
8,63304f6bb321c28505f4fb8a8e5172237775c06d996e46...,19,19,1,1,[],Ran Canetti and IBM Research: TESLA Protocol C...,"This community consists of Ran Canetti, a key ...",# Ran Canetti and IBM Research: TESLA Protocol...,7.5,The community warrants a relatively high impac...,[{'explanation': 'Ran Canetti is identified as...,"{\n ""title"": ""Ran Canetti and IBM Research:...",2026-02-19,2
9,3fb466028d2a93032f71e27ec3b52fef20e18bb32b72c3...,20,20,1,1,[],TESLA Protocol: Network Packet and Message Str...,This community centers on the TESLA protocol's...,# TESLA Protocol: Network Packet and Message S...,7.0,The impact severity is high due to TESLA's sig...,[{'explanation': 'The TESLA protocol adopts ne...,"{\n ""title"": ""TESLA Protocol: Network Packe...",2026-02-19,2


In [13]:
# ============================================================
# STEP 1: MAP PHASE
# ============================================================

map_results = run_map_phase(QUESTION, reports_at_level)

print('\n' + '=' * 60)
print('MAP RESULTS (intermediate key points from each batch):')
print('=' * 60)

for result in map_results:
    print(f"\n--- Batch {result['batch']} ---")
    for point in result['points']:
        score = point.get('score', 0)
        desc = point.get('description', 'N/A')
        bar = '█' * (score // 5)  # visual bar
        print(f"  Score: {score:3d} {bar}")
        print(f"  {desc[:200]}")
        print()

MAP PHASE: Sending 8 batches to LLM...

  Processing batch 1/8...
  Processing batch 2/8...
  Processing batch 3/8...
  Processing batch 4/8...
  Processing batch 5/8...
  Processing batch 6/8...
  Processing batch 7/8...
  Processing batch 8/8...

MAP RESULTS (intermediate key points from each batch):

--- Batch 1 ---
  Score: 100 ████████████████████
  TESLA (Timed Efficient Stream Loss-tolerant Authentication) is a cryptographic protocol specifically designed to efficiently and securely authenticate broadcast communications, particularly in lossy o

  Score: 100 ████████████████████
  TESLA provides broadcast authentication by employing symmetric cryptographic primitives, notably Message Authentication Codes (MACs), and leveraging mechanisms such as loose time synchronization, one-

  Score:  95 ███████████████████
  The protocol achieves scalability and efficiency by avoiding the need for continuous asymmetric cryptographic operations on each packet. Instead, it uses delayed key di

In [18]:
# ============================================================
# STEP 2: REDUCE PHASE
# ============================================================

final_answer, sorted_points = run_reduce_phase(QUESTION, map_results)

print('SORTED KEY POINTS (input to reduce):')
print('-' * 40)
for i, p in enumerate(sorted_points[:10]):  # Show top 10
    print(f"  {i+1}. [Score {p['score']}] (Batch {p['batch']}) {p['description'][:100]}...")

print('\n' + '=' * 60)
print('FINAL ANSWER:')
print('=' * 60)
print(final_answer)

REDUCE PHASE: 35 total points, 35 after filtering (score > 0)
  All 35 points fit within token limit (2658/8000 tokens)
SORTED KEY POINTS (input to reduce):
----------------------------------------
  1. [Score 100] (Batch 1) TESLA (Timed Efficient Stream Loss-tolerant Authentication) is a cryptographic protocol specifically...
  2. [Score 100] (Batch 1) TESLA provides broadcast authentication by employing symmetric cryptographic primitives, notably Mes...
  3. [Score 100] (Batch 2) TESLA is a cryptographic protocol designed for secure broadcast authentication, enabling the sender ...
  4. [Score 100] (Batch 2) TESLA provides broadcast authentication by structuring each message within a network packet, which i...
  5. [Score 100] (Batch 3) TESLA provides broadcast authentication by using a key chain structure, where each key is associated...
  6. [Score 100] (Batch 5) TESLA (Timed Efficient Stream Loss-Tolerant Authentication) is a protocol designed to provide secure...
  7. [Score 100]

In [15]:
# ============================================================
# PIPELINE SUMMARY
# ============================================================

total_points = sum(len(r['points']) for r in map_results)
nonzero_points = len(sorted_points)

print('Pipeline Summary')
print('=' * 40)
print(f'Community Level Used:    {COMMUNITY_LEVEL}')
print(f'Reports Processed:       {len(reports_at_level)}')
print(f'MAP Batches:             {len(map_results)}')
print(f'Total Key Points:        {total_points}')
print(f'Non-zero Key Points:     {nonzero_points}')
print(f'LLM Calls:               {len(map_results)} (map) + 1 (reduce) = {len(map_results) + 1} total')
if sorted_points:
    print(f'Top Score:               {sorted_points[0]["score"]}')
    print(f'Lowest Score:            {sorted_points[-1]["score"]}')

Pipeline Summary
Community Level Used:    1
Reports Processed:       39
MAP Batches:             8
Total Key Points:        35
Non-zero Key Points:     35
LLM Calls:               8 (map) + 1 (reduce) = 9 total
Top Score:               100
Lowest Score:            70
